In [ ]:
import torch
import librosa.display
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display
from transformers import WhisperProcessor, WhisperForConditionalGeneration

# Set the audio filename - update this to your actual filename
audio_file1 = "../datasets/whisper/taustahalina.wav"
audio_file2 = "../datasets/whisper/testi2.wav"

In [ ]:
# Load audio file
y, sr = librosa.load(audio_file1, sr=16000)
y2, sr2 = librosa.load(audio_file2, sr=16000)

# Display audio for playback
print("Play the audio:")
display(Audio(y, rate=sr))

# Print audio statistics
duration = librosa.get_duration(y=y, sr=sr)
print(f"\nAudio Statistics:")
print(f"Duration: {duration:.2f} seconds")
print(f"Sample rate: {sr} Hz")

In [ ]:
# Plot waveform
plt.figure(figsize=(14, 4))
librosa.display.waveshow(y, sr=sr)
plt.title('Audio Waveform')
plt.tight_layout()
plt.show()

# Create mel spectrogram
plt.figure(figsize=(14, 5))
S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
S_dB = librosa.power_to_db(S, ref=np.max)
librosa.display.specshow(S_dB, y_axis='mel', x_axis='time', sr=sr)
plt.colorbar(format='%+2.0f dB')
plt.title('Mel-frequency Spectrogram')
plt.tight_layout()
plt.show()

In [ ]:
def english_translation_from_transcription(transcription, model, processor, input_features):
    # Set the task to translate
    forced_decoder_ids_translate = processor.get_decoder_prompt_ids(language="fi", task="translate")

    # Generate the translation
    print("Generating English translation...")
    with torch.no_grad():
        generated_ids = model.generate(
            input_features,
            forced_decoder_ids=forced_decoder_ids_translate
        )

    # Decode the generated IDs
    translation = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]

    print("\nENGLISH TRANSLATION:")
    print("=" * 50)
    print(translation)
    marian_translation_from_transcription(transcription, model, processor, input_features)

In [ ]:
from transformers import MarianMTModel, MarianTokenizer
def marian_translation_from_transcription(transcription, model, processor, input_features):
    # Load a dedicated Finnish to English translation model
    model_name = "Helsinki-NLP/opus-mt-fi-en"
    translation_tokenizer = MarianTokenizer.from_pretrained(model_name)
    translation_model = MarianMTModel.from_pretrained(model_name)

    # Translate the Finnish transcription
    inputs = translation_tokenizer(transcription, return_tensors="pt")
    translated = translation_model.generate(**inputs)
    better_translation = translation_tokenizer.decode(translated[0], skip_special_tokens=True)

    print("BETTER ENGLISH TRANSLATION:")
    print("=" * 50)
    print(better_translation)

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import torch

audio_file = "../datasets/whisper/testi2.wav"

def transcribe_audio_with_whisper(audio_array, sr=16000, model_size="large"):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model_name = f"openai/whisper-{model_size}"

    processor = WhisperProcessor.from_pretrained(model_name)
    model = WhisperForConditionalGeneration.from_pretrained(model_name).to(device)

    forced_decoder_ids = processor.get_decoder_prompt_ids(language="fi", task="transcribe")

    input_features = processor(
        audio_array,
        sampling_rate=sr,
        return_tensors="pt"
    ).input_features.to(device)

    with torch.no_grad():
        generated_ids = model.generate(
            input_features,
            forced_decoder_ids=forced_decoder_ids
        )

    transcription = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]
    print("\nFINNISH TRANSCRIPTION:")
    print("="*50)
    print(transcription)

    english_translation_from_transcription(transcription, model, processor, input_features)

In [ ]:
transcribe_audio_with_whisper(y, sr, model_size="large")

In [ ]:
transcribe_audio_with_whisper(y, sr, model_size="base")

In [ ]:
transcribe_audio_with_whisper(y2, sr2, model_size="large")

In [ ]:
transcribe_audio_with_whisper(y2, sr2, model_size="base")